In [61]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

In [62]:
# Property names kept separately for reference
property_names = [
    "Mediterranean Avenue", "Baltic Avenue", "Oriental Avenue", "Vermont Avenue",
    "Connecticut Avenue", "St. Charles Place", "States Avenue", "Virginia Avenue",
    "St. James Place", "Tennessee Avenue", "New York Avenue", "Kentucky Avenue",
    "Indiana Avenue", "Illinois Avenue", "Atlantic Avenue", "Ventnor Avenue",
    "Marvin Gardens", "Pacific Avenue", "North Carolina Avenue", "Pennsylvania Avenue",
    "Park Place", "Boardwalk"
]

# NumPy array data
# Columns: [Rent, Rent (1 House), Rent (2 Houses), Rent (3 Houses), Rent (4 Houses), Rent (Hotel)]
rent_arr = np.array([
    [2, 10, 30, 90, 160, 250],       # Mediterranean Avenue
    [4, 20, 60, 180, 320, 450],      # Baltic Avenue
    [6, 30, 90, 270, 400, 550],      # Oriental Avenue
    [6, 30, 90, 270, 400, 550],      # Vermont Avenue
    [8, 40, 100, 300, 450, 600],     # Connecticut Avenue
    [10, 50, 150, 450, 625, 750],    # St. Charles Place
    [10, 50, 150, 450, 625, 750],    # States Avenue
    [12, 60, 180, 500, 700, 900],    # Virginia Avenue
    [14, 70, 200, 550, 750, 950],    # St. James Place
    [14, 70, 200, 550, 750, 950],    # Tennessee Avenue
    [16, 80, 220, 600, 800, 1000],   # New York Avenue
    [18, 90, 250, 700, 875, 1050],   # Kentucky Avenue
    [18, 90, 250, 700, 875, 1050],   # Indiana Avenue
    [20, 100, 300, 750, 925, 1100],  # Illinois Avenue
    [22, 110, 330, 800, 975, 1150],  # Atlantic Avenue
    [22, 110, 330, 800, 975, 1150],  # Ventnor Avenue
    [24, 120, 360, 850, 1025, 1200], # Marvin Gardens
    [26, 130, 390, 900, 1100, 1275], # Pacific Avenue
    [26, 130, 390, 900, 1100, 1275], # North Carolina Avenue
    [28, 150, 450, 1000, 1200, 1400],# Pennsylvania Avenue
    [35, 175, 500, 1100, 1300, 1500],# Park Place
    [50, 200, 600, 1400, 1700, 2000], # Boardwalk
    
    [25, 50, 100, 200, 0, 0], # Reading RR (RR ONLY HAVE 4 OPTIONS)
    [25, 50, 100, 200, 0, 0], # Pennsylvania RR
    [25, 50, 100, 200, 0, 0], # B. & O. RR
    [25, 50, 100, 200, 0, 0], # Short Line RR
    
    [1, 0, 0, 0, 0, 0], # Electric Company (RENT BASED ON DICE)
    [1, 0, 0, 0, 0, 0], # Water Works
])

# binary purachase matrix (EXLUDES RRs and Utils)
purchased_arr = np.zeros((28, 5), dtype=int)

# Cost Matrix
# Columns: [Base Purchase, +1 House, +2 Houses, +3 Houses, +4 Houses, +Hotel]
cost_arr = np.array([
    # --- 22 Streets ---
    [60,  110, 160, 210, 260, 310],  # Mediterranean Avenue
    [60,  110, 160, 210, 260, 310],  # Baltic Avenue
    [100, 150, 200, 250, 300, 350],  # Oriental Avenue
    [100, 150, 200, 250, 300, 350],  # Vermont Avenue
    [120, 170, 220, 270, 320, 370],  # Connecticut Avenue
    [140, 240, 340, 440, 540, 640],  # St. Charles Place
    [140, 240, 340, 440, 540, 640],  # States Avenue
    [160, 260, 360, 460, 560, 660],  # Virginia Avenue
    [180, 280, 380, 480, 580, 680],  # St. James Place
    [180, 280, 380, 480, 580, 680],  # Tennessee Avenue
    [200, 300, 400, 500, 600, 700],  # New York Avenue
    [220, 370, 520, 670, 820, 970],  # Kentucky Avenue
    [220, 370, 520, 670, 820, 970],  # Indiana Avenue
    [240, 390, 540, 690, 840, 990],  # Illinois Avenue
    [260, 410, 560, 710, 860, 1010], # Atlantic Avenue
    [260, 410, 560, 710, 860, 1010], # Ventnor Avenue
    [280, 430, 580, 730, 880, 1030], # Marvin Gardens
    [300, 500, 700, 900, 1100, 1300],# Pacific Avenue
    [300, 500, 700, 900, 1100, 1300],# North Carolina Avenue
    [320, 520, 720, 920, 1120, 1320],# Pennsylvania Avenue
    [350, 550, 750, 950, 1150, 1350],# Park Place
    [400, 600, 800, 1000, 1200, 1400],# Boardwalk
    
    # --- 4 Railroads ---
    # Railroads cannot be developed, so cost remains the base price
    [200, 200, 200, 200, 200, 200],  # Reading Railroad
    [200, 200, 200, 200, 200, 200],  # Pennsylvania Railroad
    [200, 200, 200, 200, 200, 200],  # B. & O. Railroad
    [200, 200, 200, 200, 200, 200],   # Short Line Railroad

    [150, 0, 0, 0, 0, 0], # Electric Company
    [150, 0, 0, 0, 0, 0] # Water Works
    
])

# Long Jail Probabilities
long_jail_props = [
    # --- 22 Streets ---
    0.020073,   # Mediterranean Avenue
    0.020369,   # Baltic Avenue
    0.021317,   # Oriental Avenue
    0.021874,   # Vermont Avenue
    0.021680,   # Connecticut Avenue
    0.025560,   # St. Charles Place
    0.021741,   # States Avenue
    0.024255,   # Virginia Avenue
    0.026802,   # St. James Place
    0.028210,   # Tennessee Avenue
    0.028118,   # New York Avenue
    0.026143,   # Kentucky Avenue
    0.025671,   # Indiana Avenue
    0.029929,   # Illinois Avenue
    0.025370,   # Atlantic Avenue
    0.025191,   # Ventnor Avenue
    0.024381,   # Marvin Gardens
    0.025236,   # Pacific Avenue
    0.024721,   # North Carolina Avenue
    0.023531,   # Pennsylvania Avenue
    0.020595,   # Park Place
    0.024832,   # Boardwalk

    # --- 4 Railroads ---
    0.028010,   # Reading Railroad
    0.026354,   # Pennsylvania Railroad
    0.028930,   # B. & O. Railroad
    0.022906,   # Short Line Railroad

    # --- 2 Utilities ---
    0.026140,   # Electric Company
    0.026507    # Water Works
]

prob_arr_temp = np.array(long_jail_props)
prob_arr = prob_arr_temp.reshape(28, 1) 


print(f"Cost Shape: {cost_arr.shape}")

print(f"Rent shape: {rent_arr.shape}")

print(f"Prob shape: {prob_arr.shape}")


expected_rent_matrix = prob_arr * rent_arr

# 3. Multiply by the binary variables X and sum everything up
# (This multiplies each cell of the expected_rent_matrix by the corresponding X variable)


Cost Shape: (28, 6)
Rent shape: (28, 6)
Prob shape: (28, 1)


In [63]:
m = gp.Model("model")
m.Params.OutputFlag = 0 

x1 = m.addMVar(shape=(28, 6), lb=0, vtype=GRB.BINARY, name="rent")
x2 = m.addMVar(shape=(28, 6), lb=0, vtype=GRB.BINARY, name="purchase")
x3 = m.addMVar(shape=(28, 6), lb=0, vtype=GRB.BINARY, name="cost")

In [64]:

# 3. Multiply by the binary variables X and sum everything up
# (This multiplies each cell of the expected_rent_matrix by the corresponding X variable)
objective = (expected_rent_matrix * x2).sum()

m.setObjective(objective, GRB.MAXIMIZE)

In [65]:
m.addConstr(
    gp.quicksum(cost_arr[i, j] * x2[i, j] for i in range(22) for j in range(6)) <= 4500, 
    name="Budget"
)

<MConstr ()>

In [67]:
m.optimize()
print(m.NumConstrs)  # Should show how many constraints you added

1


In [70]:
# --- PRINTOUT ---
print("\n" + "="*60)
print("MONOPOLY PURCHASE OPTIMIZATION RESULTS")
print("="*60)

# 1. Show the Objective Value (Total Expected Rent based on probabilities)
print(f"Total Expected Revenue Score: {m.ObjVal:.4f}")
print("-" * 60)

# 2. Extract the raw solved matrix (0s and 1s)
# X.X returns the values of the MVar as a numpy array
solved_matrix = x2.X

total_spent = 0
purchases = []

# 3. Loop through the matrix to find what was bought
for i in range(22):
    for j in range(6):
        if solved_matrix[i, j] > 0.5: # If the solver set the binary variable to 1
            cost = cost_arr[i, j]
            total_spent += cost
            
            # Label the development level
            if j == 0: level = "Base Property"
            elif j == 5: level = "Hotel"
            else: level = f"{j} House(s)"
            
            purchases.append(f"{property_names[i]:<25} | {level:<15} | Cost: ${cost}")

# 4. Print the budget summary
print(f"Total Money Spent: ${total_spent} / $4500")
print(f"Remaining Budget:  ${4500 - total_spent}")
print("-" * 60)

# 5. Print the itemized list of what to buy
print(f"{'Property':<25} | {'Development':<15} | {'Cost'}")
print("-" * 60)
for p in purchases:
    print(p)
print("="*60)


MONOPOLY PURCHASE OPTIMIZATION RESULTS
Total Expected Revenue Score: 206.2048
------------------------------------------------------------
Total Money Spent: $4500 / $4500
Remaining Budget:  $0
------------------------------------------------------------
Property                  | Development     | Cost
------------------------------------------------------------
Baltic Avenue             | Base Property   | Cost: $60
St. James Place           | Hotel           | Cost: $680
Tennessee Avenue          | 4 House(s)      | Cost: $580
Tennessee Avenue          | Hotel           | Cost: $680
New York Avenue           | 4 House(s)      | Cost: $600
New York Avenue           | Hotel           | Cost: $700
Boardwalk                 | 4 House(s)      | Cost: $1200
